# Day 2 — Understanding Attention in Transformers

## What this covers
How the Transformer model uses **Attention** to decide which earlier words are important when processing a word like "it".

## Setup

In [ ]:
!pip install transformers torch --quiet

## Load model with attention output

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)

## Configuration

In [ ]:
text = "The firewall blocked the traffic because it was suspicious"
target_word = "it"   # Change this to any word you want to analyze

## Run the model

In [ ]:
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
attentions = outputs.attentions
last_layer_attn = attentions[-1][0].mean(dim=0)  # Average all heads

## Find target word and show attention

In [ ]:
# Find position of target word
try:
    target_idx = tokens.index("Ġ" + target_word)
except ValueError:
    target_idx = tokens.index(target_word)

print(f"Attention from '{tokens[target_idx]}' (position {target_idx}) to previous tokens:\n")

for i, tok in enumerate(tokens):
    if i > target_idx:
        continue
    weight = last_layer_attn[target_idx, i].item()
    if weight > 0.01:
        print(f"  {tok:15s} → {weight:.4f}")

## Query, Key, Value Concept

- **Query** = "What am I looking for?" → The word "it"
- **Key** = "What do I represent?" → Label of each previous word
- **Value** = "What information do I give?" → The actual meaning/content